In [3]:

# In this homework we won't use the same approach for embedding as in the module. 
# That is, we won't use the sentence-transformers library. 
# Instead, we'll use the lightweight embedding approach with the ONNX Embedder.

# # Both approaches produce identical vectors, but the ONNX runtime is far lighter. 
# It needs no PyTorch and no CUDA, which makes the installation about 30x smaller and lets it run anywhere, 
# including a basic Codespace. We skimmed through it in the lesson and said we'd cover it in the homework - so here we are.

# We prepare the environment the same way as in the module's ONNX Runtime lesson.

# Create a fresh project and install the dependencies:

# mkdir llm-zoomcamp-hw2 && cd llm-zoomcamp-hw2
# uv init --no-workspace
# uv add onnxruntime tokenizers numpy tqdm minsearch gitsource
# uv add --dev huggingface-hub jupyter
# We also need two helper scripts from the embed/ directory of the course repo:

# download.py (fetches an ONNX model from HuggingFace) and
# embedder.py (the Embedder class with an encode interface)
# Let's download them:

# PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed
# !wget $PREFIX/download.py
# !wget $PREFIX/embedder.py

# DK - in notebook I can use pythonic way wget, not bash - style :
# PREFIX = "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed"

# !wget {PREFIX}/download.py
# !wget {PREFIX}/embedder.py

# By default download.py fetches Xenova/all-MiniLM-L6-v2, the ONNX version of the all-MiniLM-L6-v2 model from the lessons:

# uv run python download.py
# Now we're ready to do the homework.


In [6]:
# Q1. Embedding a query
# Embed the following query:

# How does approximate nearest neighbor search work?

# The embedder returns a vector of 384 numbers. What's the first value (v[0])?

# -0.31
# -0.02 == np.float64(-0.02058203437252893)
# 0.12
# 0.44

from embedder import Embedder

embed = Embedder()

q1 = "How does approximate nearest neighbor search work?"
v1 = embed.encode(q1)
v1[0]

np.float64(-0.02058203437252893)

In [7]:
# Loading the data
# We pull the lesson pages from the course repository, the same way as in homework 1. 
# We pin to commit 8c1834d so everyone works with the same data.

from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

# Each document is a dictionary with a filename and content, and there are 72 pages.

documents[10]

{'content': "# Agents\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=6uG4_Ivv60E&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn Part 1 of this module we built RAG pipelines.\n\nEvery pipeline we wrote followed the same flow:\n\n- search the FAQ,\n- build a prompt with the results,\n- send it to the LLM.\n\nThis returns good answers when the user's query matches the documents.\nThe search finds the right entry, the LLM reads it, and you get a\nhelpful reply.\n\nOften, though, the search returns nothing useful.\n\n- Maybe the user made a typo.\n- Maybe they asked the question in an unusual way.\n- Maybe they need information from two different searches.\n\nWe use lexical search here, so the search looks for an exact match.\nOne typo and it misses the entry it needed. In our pipeline there's\nno recovery. The search runs once, and if it returns garbage the LLM\ngets garbage. Our pipeline always does the same thing, no matter what.\n\nInstead of routing the user question str

In [9]:
documents[10]['filename']

'01-agentic-rag/lessons/11-agents-intro.md'

In [12]:
# Q2. Cosine similarity
# The embedder returns normalized vectors, so the dot product between two of them is their cosine similarity.

# Take the page 02-vector-search/lessons/07-sqlitesearch-vector.md, embed its content, and compute the cosine similarity with the query vector from Q1. What do you get?

# 0.07
# 0.37 -- np.float64(0.36107026789538205)
# 0.68
# 0.92

# STEP 1 - Take the page 02-vector-search/lessons/07-sqlitesearch-vector.md
from tqdm.auto import tqdm

for d in tqdm(documents):
    if d['filename'] == '02-vector-search/lessons/07-sqlitesearch-vector.md':
        content = d['content']

# STEP 2 - embed its content
dv = embed.encode(content)

# STEP 3 - compute the cosine similarity with the query vector from Q1. What do you get?
similarity = v1.dot(dv)
similarity


  0%|          | 0/72 [00:00<?, ?it/s]

np.float64(0.36107026789538205)